# Demo A — The autoencoder and PCA

**Class 11 · A1–A2** (slides: `slides/11_autoencoders.md`)

* **A1:** a linear autoencoder (no activation, squared error) learns the PCA subspace of Class 09: same reconstruction error, principal angles near zero. A by-hand example with three points.
* **A2:** a nonlinear autoencoder (ReLU) on the digits: the code size against the reconstruction error (against PCA), reconstructions, and the two-dimensional codes.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import polars as pl
from scipy.linalg import subspace_angles
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from faa import demo, plotlib, semisup
from faa.demo import show, slide

demo.setup("11")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
X, y = semisup.digits_flat()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

## Slide: One Hidden Unit by Hand (A1)

Three points in the plane, $x_1 = (2, 1)$, $x_2 = (-1, -0.5)$, $x_3 = (0, 0)$, and a linear autoencoder with one code unit: $z = w^\top x$ and $\hat x = w z$ with $w = (0.8, 0.6)$ (a unit vector near the direction of the points).
Reconstruction error of each point $\lVert x - \hat x \rVert^2$ and the mean.

In [ ]:
P3 = np.array([[2.0, 1.0], [-1.0, -0.5], [0.0, 0.0]])
w = np.array([0.8, 0.6])
z = P3 @ w
Xh = np.outer(z, w)
err = ((P3 - Xh) ** 2).sum(1)
slide("One Hidden Unit by Hand")
print(f"codes z = {np.round(z, 3).tolist()}; reconstructions {np.round(Xh, 3).tolist()}")
print(f"errors per point {np.round(err, 4).tolist()}; mean {err.mean():.4f}")
u = np.linalg.svd(P3, full_matrices=False)[2][0]
err_best = ((P3 - np.outer(P3 @ u, u)) ** 2).sum(1)
print(f"first right singular vector {np.round(u, 3).tolist()}: mean error {err_best.mean():.5f}")

## Slide: A Linear Autoencoder Finds the PCA Subspace (A1)

Linear autoencoder $64 \to k \to 64$ (Keras, Adam, 300 epochs of the 1257 training digits) against PCA with $k$ components.
Error = mean squared error per pixel on the 540 test digits. Angle = largest principal angle between the span of the decoder columns and the span of the $k$ PCA directions (degrees); `eigen_ratio` = $\lambda_k / \lambda_{k+1}$, the gap after the last kept direction: near 1 the subspace is ill-defined.

In [ ]:
rows = []
models = {}
for k in (2, 5, 10, 20):
    ae, enc = semisup.autoencoder(64, k, activation="linear", seed=0)
    semisup.fit_ae(ae, Xtr, epochs=300, batch=128)
    pca = PCA(k).fit(Xtr)
    mse_pca = float(np.mean((pca.inverse_transform(pca.transform(Xte)) - Xte) ** 2))
    mse_ae = float(np.mean((np.asarray(ae.predict(Xte, verbose=0)) - Xte) ** 2))
    dec = np.asarray(ae.layers[-1].get_weights()[0]).T  # (64, k): the columns span the decoder subspace
    ang = float(np.degrees(subspace_angles(dec, pca.components_.T).max()))
    ev = PCA(k + 1).fit(Xtr).explained_variance_
    rows.append(
        {
            "k": k,
            "pca_mse": mse_pca,
            "ae_mse": mse_ae,
            "ratio": mse_ae / mse_pca,
            "angle_deg": ang,
            "eigen_ratio": ev[k - 1] / ev[k],
        }
    )
    models[k] = (ae, enc, pca)
lin = pl.DataFrame(rows)
slide("A Linear Autoencoder Finds the PCA Subspace")
show(lin, 4)

fig, ax = plotlib.fig("col", 1.6)
ax.plot(lin["k"], lin["pca_mse"], color=BLUE, marker="o", ms=3, label="PCA")
ax.plot(lin["k"], lin["ae_mse"], color=ORANGE, marker="s", ms=3, ls="--", label="linear AE")
ax.set_xlabel("code size $k$")
ax.set_ylabel("test MSE per pixel")
ax.legend()
plotlib.save(fig, "a1_linear_pca")

## Slide: Where the Autoencoder Differs from PCA (A1)

The codes of the linear AE are not the PCA scores: any invertible mixing of the code gives the same reconstruction. Correlation of each PCA score with its best linear fit from the AE code (k = 5): the span is the same, the axes are not.

In [ ]:
ae, enc, pca = models[5]
Za = np.asarray(enc.predict(Xte, verbose=0))
Zp = pca.transform(Xte)
A = np.c_[Za, np.ones(len(Za))]
r2 = []
for j in range(5):
    coef = np.linalg.lstsq(A, Zp[:, j], rcond=None)[0]
    r2.append(1 - np.sum((A @ coef - Zp[:, j]) ** 2) / np.sum((Zp[:, j] - Zp[:, j].mean()) ** 2))
direct = [abs(np.corrcoef(Za[:, j], Zp[:, j])[0, 1]) for j in range(5)]
slide("Where the Autoencoder Differs from PCA")
print("R^2 of each PCA score from the 5 AE codes:", np.round(r2, 4).tolist())
print("|correlation| of AE code j with PCA score j:", np.round(direct, 3).tolist())

## Slide: Depth and Non-Linearity (A2)

Nonlinear autoencoder $64 \to 32 \to k \to 32 \to 64$ (ReLU, sigmoid output), same training, 3 seeds. Test error against PCA for the same $k$.

In [ ]:
rows = []
for k in (2, 4, 8, 16, 32):
    pca = PCA(k).fit(Xtr)
    mse_pca = float(np.mean((pca.inverse_transform(pca.transform(Xte)) - Xte) ** 2))
    errs = []
    for seed in range(3):
        ae, enc = semisup.autoencoder(64, k, hidden=(32,), seed=seed)
        semisup.fit_ae(ae, Xtr, epochs=150, batch=64)
        errs.append(float(np.mean((np.asarray(ae.predict(Xte, verbose=0)) - Xte) ** 2)))
        if seed == 0:
            models[("deep", k)] = (ae, enc)
    rows.append({"k": k, "pca_mse": mse_pca, "ae_mse": np.mean(errs), "ae_std": np.std(errs)})
deep = pl.DataFrame(rows)
slide("Depth and Non-Linearity")
show(deep, 4)

fig, ax = plotlib.fig("col", 1.6)
ax.plot(deep["k"], deep["pca_mse"], color=BLUE, marker="o", ms=3, label="PCA")
ax.errorbar(deep["k"], deep["ae_mse"], deep["ae_std"], color=ORANGE, marker="s", ms=3, capsize=2, label="ReLU AE")
ax.set_xlabel("code size $k$")
ax.set_ylabel("test MSE per pixel")
ax.set_xscale("log", base=2)
ax.legend()
plotlib.save(fig, "a2_code_size")

## Slide: Reconstructions (A2)

First row: eight test digits. Then the reconstructions with $k = 2, 8, 32$ (ReLU autoencoder).

In [ ]:
import matplotlib.pyplot as plt

idx = np.array([np.flatnonzero(yte == c)[0] for c in range(8)])
fig, axes = plt.subplots(4, 8, figsize=(plotlib.TEXT_WIDTH if hasattr(plotlib, "TEXT_WIDTH") else 4.25, 2.2))
rowsimg = [("input", Xte[idx])]
for k in (2, 8, 32):
    ae, _ = models[("deep", k)]
    rowsimg.append((f"$k = {k}$", np.asarray(ae.predict(Xte[idx], verbose=0))))
for r, (name, imgs) in enumerate(rowsimg):
    for c in range(8):
        a = axes[r, c]
        a.imshow(imgs[c].reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
        a.set_xticks([])
        a.set_yticks([])
        a.grid(False)
        if c == 0:
            a.set_ylabel(name, fontsize=6.5)
fig.subplots_adjust(wspace=0.05, hspace=0.05)
plotlib.save(fig, "a2_reconstructions")
slide("Reconstructions")
print(
    "test MSE per pixel on the eight digits:",
    {name: round(float(np.mean((imgs - Xte[idx]) ** 2)), 4) for name, imgs in rowsimg[1:]},
)

## Slide: Two-Dimensional Codes (A2)

The 540 test digits in the code plane of the PCA and of the autoencoder ($k = 2$), coloured by digit (the colours were not used in training).
Separation measured by the accuracy of a 5-nearest-neighbour classifier on the two coordinates (fitted on the training codes).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

pca2 = PCA(2).fit(Xtr)
ae2, enc2 = models[("deep", 2)]
Zs = {
    "PCA": (pca2.transform(Xtr), pca2.transform(Xte)),
    "autoencoder": (np.asarray(enc2.predict(Xtr, verbose=0)), np.asarray(enc2.predict(Xte, verbose=0))),
}
slide("Two-Dimensional Codes")
fig, axes = plt.subplots(1, 2, figsize=(4.25, 1.9), sharey=False)
for ax, (name, (Ztr, Zte)) in zip(axes, Zs.items(), strict=True):
    acc = KNeighborsClassifier(5).fit(Ztr, ytr).score(Zte, yte)
    print(f"{name}: 5-NN accuracy on the 2-D code {acc:.3f}")
    ax.scatter(Zte[:, 0], Zte[:, 1], c=yte, cmap="tab10", s=4)
    ax.set_title(f"{name} (5-NN {acc:.2f})", fontsize=7)
    ax.set_xlabel("code 1")
    ax.set_ylabel("code 2")
fig.tight_layout()
plotlib.save(fig, "a2_codes_2d")

## Failure case: a code as large as the input

With $k = 64$ and ReLU the autoencoder can copy the input: the error is tiny, but the code has learned nothing useful. Test MSE for the sizes of the code and the number of training epochs (3 epochs: nothing learned yet).

In [ ]:
rows = []
for k, ep in ((64, 150), (64, 3), (8, 3)):
    ae, _ = semisup.autoencoder(64, k, hidden=(32,), seed=0)
    semisup.fit_ae(ae, Xtr, epochs=ep, batch=64)
    rows.append({"k": k, "epochs": ep, "test_mse": float(np.mean((np.asarray(ae.predict(Xte, verbose=0)) - Xte) ** 2))})
slide("A code as large as the input")
show(pl.DataFrame(rows), 4)